In [ ]:
%%capture
!pip install unsloth
!pip install transformers==4.55.4
!pip install wandb

In [2]:
from unsloth import FastModel
import torch
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer,SFTConfig
from transformers import TrainingArguments
from huggingface_hub import notebook_login
import os


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-09-15 05:54:42.741316: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757915682.764080     972 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757915682.771048     972 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
import wandb
wandb.login()

# notebook_login()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rakeshthalaboina11 (rakeshthalaboina11-jb-institute-of-engineering-and-techn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:

os.environ["WANDB_PROJECT"]="Qwen3_4B_NCRT_Physics_12th_Finetuned"
os.environ["WANDB_LOG_MODEL"] = "checkpoint"

os.environ["WANDB_NAME"] = "qwen3-4B"
os.environ["WANDB_WATCH"] = "all"



In [5]:
from unsloth import FastLanguageModel
import torch


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Thinking-2507",
    max_seq_length = 2048,  
    load_in_4bit = True,
     )

==((====))==  Unsloth 2025.9.4: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128,           
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 128,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2025.9.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [7]:
from datasets import load_dataset
ds=load_dataset("KadamParth/NCERT_Physics_12th")

In [8]:
def generate_conversation(examples):
    question = examples["Question"]
    answer = examples["Answer"]
    explanation = examples["Explanation"]

    final = []

    for q, a, e in zip(question, answer, explanation):
        final_entry = [
            {"role": "system", "content": "Follow these steps to answer the user's question:\n1. First, reason step-by-step inside <think> tags. This is your internal monologue.\n2. After the </think> tag, provide a complete, helpful, and polished answer for the user that includes both a direct answer and an explanation."},
            {"role": "user", "content": q},
            {"role": "assistant", "content": f"Answer: {a}\nExplanation: {e}"}
        ]
        final.append(final_entry)

    return {"final": final}
ds=ds.map(generate_conversation, batched=True)

In [9]:
ds['train']['final'][10]

[{'content': "Follow these steps to answer the user's question:\n1. First, reason step-by-step inside <think> tags. This is your internal monologue.\n2. After the </think> tag, provide a complete, helpful, and polished answer for the user that includes both a direct answer and an explanation.",
  'role': 'system'},
 {'content': 'When a glass rod is rubbed with wool, what type of charge do each of the materials acquire?',
  'role': 'user'},
 {'content': '<think>Electrostatics is the study of electrical charges at rest. When certain materials, like glass and wool, are rubbed together, they acquire an electric charge and can attract or repel other charged objects. This phenomenon is due to the transfer of electrons between materials, causing an imbalance of charge.</think>\nAnswer: The glass rod acquires a positive charge, and the wool acquires a negative charge.\nExplanation: Electrostatics is the study of electrical charges at rest. When certain materials, like glass and wool, are rubbe

In [10]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-thinking",
)

In [11]:
def formatting_prompts_func(examples):
   convos = examples["final"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
   return { "text" : texts, }

dataset = ds.map(formatting_prompts_func, batched = True)

In [12]:
dataset['train'][100]['text']

"<|im_start|>system\nFollow these steps to answer the user's question:\n1. First, reason step-by-step inside <think> tags. This is your internal monologue.\n2. After the </think> tag, provide a complete, helpful, and polished answer for the user that includes both a direct answer and an explanation.<|im_end|>\n<|im_start|>user\nIf 10^10 electrons move from one body to another every second, how long will it take to transfer 1 C of charge?<|im_end|>\n<|im_start|>assistant\n<think>\nElectrons carry a fundamental unit of charge, which is measured in Coulombs (C). The charge of one electron is approximately 1.6 × 10^-19 C. To calculate the time required to accumulate a certain amount of electric charge, you need to understand the rate at which electrons are transferred. This rate is often given as current (I), measured in Amperes (A), where 1 Ampere equals 1 Coulomb per second (1 A = 1 C/s). For example, if you have a current of 2 Amperes, it means 2 Coulombs of charge are flowing per secon

In [13]:
dataset=dataset['train'].train_test_split(test_size=0.1,seed=42)

In [14]:
dataset['train']['text'][0]

"<|im_start|>system\nFollow these steps to answer the user's question:\n1. First, reason step-by-step inside <think> tags. This is your internal monologue.\n2. After the </think> tag, provide a complete, helpful, and polished answer for the user that includes both a direct answer and an explanation.<|im_end|>\n<|im_start|>user\nWhat is the formula for electric flux (Î¦E) according to Gauss's Law?<|im_end|>\n<|im_start|>assistant\n<think>\nGauss's Law relates the electric flux through a closed surface to the charge enclosed within that surface. It is mathematically expressed as Î¦E = Q/Îµ0, where Î¦E is the electric flux, Q is the charge enclosed, and Îµ0 is the permittivity of free space. When the charge Q changes with time, it produces a current i = dQ/dt, leading to a changing electric flux. This changing electric flux implies a displacement current, which is a crucial concept in electromagnetism, particularly in the generalization of Ampere's circuital law.\n</think>\n\nAnswer: The 

In [15]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'],
    args = SFTConfig(
        output_dir="./Qwen3_4B_NCRT_Physics_12th_Finetuned",
        dataset_text_field = "text",
        per_device_train_batch_size = 8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps = 2,
        warmup_steps=5,
        weight_decay=0.05,
        num_train_epochs=1,
        learning_rate=2e-5,
        eval_strategy="steps",
        save_strategy="steps",
        report_to="wandb",
        save_steps=30,
        max_steps=100,
        logging_steps=10,
        logging_dir="./logs",
        logging_first_step=True,
        optim='adamw_8bit',
        lr_scheduler_type = "cosine",
   ),
)

In [ ]:

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
                                    )
trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,715 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 264,241,152 of 4,286,709,248 (6.16% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
10,0.816000,0.551215
20,0.501900,0.457432
30,0.458100,0.424137
40,0.418700,0.406964
50,0.417300,0.395209
60,0.404300,0.387955
70,0.396200,0.383678
80,0.403900,0.381539
90,0.392800,0.380669
100,0.404300,0.380510


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
wandb: Adding directory to artifact (./Qwen3_4B_NCRT_Physics_12th_Finetuned/checkpoint-30)... Done. 4.0s
wandb: Adding directory to artifact (./Qwen3_4B_NCRT_Physics_12th_Finetuned/checkpoint-60)... Done. 4.1s
wandb: Adding directory to artifact (./Qwen3_4B_NCRT_Physics_12th_Finetuned/checkpoint-90)... Done. 4.1s
wandb: Adding directory to artifact (./Qwen3_4B_NCRT_Physics_12th_Finetuned/checkpoint-100)... Done. 4.2s


TrainOutput(global_step=100, training_loss=0.4639714550971985, metrics={'train_runtime': 4850.3362, 'train_samples_per_second': 0.66, 'train_steps_per_second': 0.021, 'total_flos': 3.717782914218394e+16, 'train_loss': 0.4639714550971985, 'epoch': 0.6779661016949152})

In [20]:
import torch
torch.cuda.empty_cache()


In [ ]:
wandb.finish()

In [ ]:
messages = [
    {"role" : "user", "content" : "If 10^10 electrons move from one body to another every second, how long will it take to transfer 1 C of charge?"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, 
    enable_thinking = False, 
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 4500, 
    temperature = 0.7, top_p = 0.8, top_k = 20, 
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

<|im_start|>user
If 10^10 electrons move from one body to another every second, how long will it take to transfer 1 C of charge?<|im_end|>
<|im_start|>assistant
<think>
Okay, the user is asking how long it takes to transfer 1 coulomb of charge if 10^10 electrons move from one body to another every second. Let me start by recalling the basics. Charge is measured in coulombs, and one electron has a charge of about 1.6 × 10^-19 coulombs. So, if I can figure out how much charge moves per second, I can find out how long it takes to get 1 coulomb.

First, the rate is given as 10^10 electrons per second. Each electron carries a charge of 1.6 × 10^-19 C. So, the total charge per second would be 10^10 multiplied by 1.6 × 10^-19. Let me calculate that: 10^10 × 1.6 × 10^-19 = 1.6 × 10^-9 C/s. That means the charge transfer rate is 1.6 nanocoulombs per second.

Now, to find the time to transfer 1 coulomb, I divide the total charge by the rate. So, time = 1 C / (1.6 × 10^-9 C/s). That gives me 6.25